# Gold Layer - Star Schema Modeling

In [0]:
# ============================================================
# Notebook : 03_gold_layer
# Project  : Supply Chain Control Tower
# Author   : Nisha Sorallikar
# Purpose  : Build Gold layer dimension and fact tables
#            using a Star Schema for analytics.
# ============================================================

In [0]:
# ---------------------------------------
# Step 1 : Read Silver Delta Table
# ---------------------------------------

df_gold = spark.table("dev_project.default.silver_supply_chain")

df_gold.show(5)

from pyspark.sql.functions import countDistinct

df_gold.groupBy("order_id") \
       .agg(countDistinct("order_item_id").alias("items_per_order")) \
       .orderBy("items_per_order", ascending=False) \
       .show(10)

       # ---------------------------------------
# Step 2 : Create Customer Dimension
# ---------------------------------------

# Select only the customer-related columns
# from the Silver table.

dim_customer = df_gold.select(
    "customer_id",
    "customer_fname",
    "customer_lname",
    "customer_city",
    "customer_state",
    "customer_country",
    "customer_zipcode",
    "customer_segment"
)

# ---------------------------------------
# Step 3 : Remove Duplicate Customers
# ---------------------------------------

# Keep only one record for each customer.

dim_customer = dim_customer.dropDuplicates(["customer_id"])

# ---------------------------------------
# Step 4 : Validate Customer Dimension
# ---------------------------------------

print(f"Total Customers : {dim_customer.count()}")

dim_customer.show(10, truncate=False)

# ---------------------------------------
# Step 5 : Create Customer Dimension Table
# ---------------------------------------

# Save the customer dimension as a Delta table
# in the Gold layer.

(
    dim_customer.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("dev_project.default.dim_customer")
)

# ---------------------------------------
# Step 6 : Verify Customer Dimension
# ---------------------------------------

spark.sql("""
SELECT COUNT(*) AS total_customers
FROM dev_project.default.dim_customer
""").show()

# ---------------------------------------
# Step 7 : Create Product Dimension
# ---------------------------------------

# Select only product-related columns.

dim_product = df_gold.select(
    "product_card_id",
    "product_category_id",
    "product_name",
    "product_price",
    "category_name",
    "department_name"
)

# ---------------------------------------
# Step 8 : Remove Duplicate Products
# ---------------------------------------

dim_product = dim_product.dropDuplicates(["product_card_id"])

# ---------------------------------------
# Step 9 : Validate Product Dimension
# ---------------------------------------

print(f"Total Products : {dim_product.count()}")

dim_product.show(10, truncate=False)

# ---------------------------------------
# Step 10 : Save Product Dimension
# ---------------------------------------

# Save the Product Dimension
# into the Gold Layer.

(
    dim_product.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("dev_project.default.dim_product")
)

# ---------------------------------------
# Step 11 : Verify Product Dimension
# ---------------------------------------

spark.sql("""
SELECT COUNT(*) AS total_products
FROM dev_project.default.dim_product
""").show()

# ============================================================
# Step 12 : Create Date Dimension
# ============================================================

from pyspark.sql.functions import (
    col,
    to_date,
    year,
    quarter,
    month,
    dayofmonth,
    date_format
)

# ---------------------------------------
# Step 12.1 : Select Order Date
# ---------------------------------------

# Select the order_date column from the Gold DataFrame
dim_date = df_gold.select("order_date")


# ---------------------------------------
# Step 12.2 : Convert Timestamp to Date
# ---------------------------------------

# Convert timestamp to date (remove time portion)
dim_date = dim_date.withColumn(
    "date",
    to_date(col("order_date"))
)


# ---------------------------------------
# Step 12.3 : Keep Only Unique Dates
# ---------------------------------------

# Remove duplicate calendar dates
dim_date = dim_date.select("date").dropDuplicates()


# ---------------------------------------
# Step 12.4 : Create Calendar Attributes
# ---------------------------------------

dim_date = (
    dim_date
    .withColumn("year", year(col("date")))
    .withColumn("quarter", quarter(col("date")))
    .withColumn("month", month(col("date")))
    .withColumn("month_name", date_format(col("date"), "MMMM"))
    .withColumn("day", dayofmonth(col("date")))
    .withColumn("day_name", date_format(col("date"), "EEEE"))
)


# ---------------------------------------
# Step 12.5 : Validate Date Dimension
# ---------------------------------------

print(f"Total Dates : {dim_date.count()}")

dim_date.orderBy("date").show(10, truncate=False)


# ---------------------------------------
# Step 12.6 : Verify Schema
# ---------------------------------------

dim_date.printSchema()

# ---------------------------------------
# Step 13 : Save Date Dimension
# ---------------------------------------

(
    dim_date.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("dev_project.default.dim_date")
)

# ---------------------------------------
# Step 14 : Verify Date Dimension
# ---------------------------------------

spark.sql("""
SELECT COUNT(*) AS total_dates
FROM dev_project.default.dim_date
""").show()

# ---------------------------------------
# Step 15 : Create Shipping Dimension
# ---------------------------------------

# Select shipping-related descriptive columns

dim_shipping = df_gold.select(
    "shipping_mode",
    "delivery_status"
)

# ---------------------------------------
# Step 16 : Remove Duplicate Shipping Records
# ---------------------------------------

dim_shipping = dim_shipping.dropDuplicates()

# ---------------------------------------
# Step 17 : Validate Shipping Dimension
# ---------------------------------------

print(f"Total Shipping Records : {dim_shipping.count()}")

dim_shipping.show(truncate=False)

# ---------------------------------------
# Step 18 : Save Shipping Dimension
# ---------------------------------------

(
    dim_shipping.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("dev_project.default.dim_shipping")
)

# ---------------------------------------
# Step 19 : Verify Shipping Dimension
# ---------------------------------------

spark.sql("""
SELECT COUNT(*) AS total_shipping_records
FROM dev_project.default.dim_shipping
""").show()

# ============================================================
# Step 20 : Create Fact Orders
# ============================================================

fact_orders = df_gold.select(

    # -----------------------------
    # Transaction Keys
    # -----------------------------
    "order_item_id",
    "order_id",

    # -----------------------------
    # Foreign Keys
    # -----------------------------
    "customer_id",
    "product_card_id",
    "order_date",
    "shipping_mode",
    "delivery_status",

    # -----------------------------
    # Measures
    # -----------------------------
    "sales",
    "order_item_quantity",
    "order_item_total",
    "order_item_discount",
    "order_item_discount_rate",
    "order_profit_per_order",
    "benefit_per_order",
    "days_for_shipping_real",
    "days_for_shipment_scheduled",
    "late_delivery_risk"
)
# ---------------------------------------
# Step 21 : Validate Fact Table
# ---------------------------------------

print(f"Total Fact Records : {fact_orders.count()}")

fact_orders.show(10, truncate=False)

fact_orders.printSchema()

# ---------------------------------------
# Step 22 : Save Fact Table
# ---------------------------------------

(
    fact_orders.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("dev_project.default.fact_orders")
)

# ---------------------------------------
# Step 23 : Verify Fact Table
# ---------------------------------------

spark.sql("""
SELECT COUNT(*) AS total_fact_records
FROM dev_project.default.fact_orders
""").show()

# ---------------------------------------
# Step 22 : Save Fact Table
# ---------------------------------------

(
    fact_orders.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("dev_project.default.fact_orders")
)
# ---------------------------------------
# Step 23 : Verify Fact Table
# ---------------------------------------

spark.sql("""
SELECT COUNT(*) AS total_fact_records
FROM dev_project.default.fact_orders
""").show()